# Cats vs. Dogs: Simple PyTorch CNN

A clean, minimal, easy-to-understand Convolutional Neural Network built from scratch in PyTorch.

### Pipeline:
1. **Setup & Data**: Load Kaggle Cats vs. Dogs with standard 128x128 resize.
2. **Simple Architecture**: 2 scanning rounds (Conv -> ReLU -> MaxPool) + 1 voting layer.
3. **Train**: 10 fast epochs with Adam & CrossEntropyLoss.
4. **Inference**: Predict on any test photo with confidence percentage.

In [ ]:
# Step 1: Imports & Device Setup
from pathlib import Path
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image

# Device: Apple Silicon GPU (mps), Nvidia GPU (cuda), or CPU
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")

In [ ]:
# Step 2: Load and Prepare Data
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset" / "cats_and_dogs"

# Simple transformations: Resize to 128x128 and convert to PyTorch Tensor
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load images from folders
train_dataset = datasets.ImageFolder(str(DATASET_DIR / "train"), transform=transform)
val_dataset = datasets.ImageFolder(str(DATASET_DIR / "validation"), transform=transform)

# Fast 2,000-image subset for quick training
SUBSET_SIZE = 2000
train_sub = Subset(train_dataset, random.sample(range(len(train_dataset)), SUBSET_SIZE))
val_sub = Subset(val_dataset, random.sample(range(len(val_dataset)), SUBSET_SIZE // 4))

# DataLoaders: batches of 32 images
train_loader = DataLoader(train_sub, batch_size=32, shuffle=True)
val_loader = DataLoader(val_sub, batch_size=32, shuffle=False)

class_names = train_dataset.classes # ['cats', 'dogs']
print("Classes:", class_names)
print(f"Training set: {len(train_sub)} images ({len(train_loader)} batches)")
print(f"Validation set: {len(val_sub)} images ({len(val_loader)} batches)")

In [ ]:
# Step 3: Simple CNN Architecture
class SimpleCatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # PART 1: Find Clues (2 simple scanning rounds)
        self.features = nn.Sequential(
            # Round 1: 16 filters | 128x128 -> 64x64
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Round 2: 32 filters | 64x64 -> 32x32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # PART 2: Make the Final Guess
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 2)  # Vote: Cat (0) or Dog (1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCatDogCNN().to(device)
print("Simple CNN Architecture:")
print(model)

In [ ]:
# Step 4: Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Ready to train!")

In [ ]:
# Step 5: Training Loop
NUM_EPOCHS = 6
print(f"Training Simple CNN for {NUM_EPOCHS} epochs on {device}...")

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Training Phase ---
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    train_loss = total_loss / total
    train_acc = (correct / total) * 100
    
    # --- Validation Phase ---
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    val_acc = (val_correct / val_total) * 100
    print(f"Epoch [{epoch:2d}/{NUM_EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("Training Complete!")


In [ ]:
# Step 6: Test on Individual Images (Inference)
def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        pred_idx = torch.argmax(probabilities).item()
        
    label = class_names[pred_idx]
    confidence = probabilities[pred_idx].item() * 100
    print(f"{Path(image_path).name} -> {label.upper()} ({confidence:.1f}% confidence)")

# Test 1 Cat and 1 Dog from validation set
test_cat = list((DATASET_DIR / "validation" / "cats").glob("*.jpg"))[0]
test_dog = list((DATASET_DIR / "validation" / "dogs").glob("*.jpg"))[0]

print("--- Predictions on Unseen Photos ---")
predict(test_cat)
predict(test_dog)